## Single-tile MCF fiber assignment using the original input mock (without pre-selecting decollided targets). --09-07-2026

In [1]:
import argparse
import glob
import gc
import os
import signal
import sys
import time
from collections import defaultdict

import healpy as hp
import numpy as np
from astropy.table import Table
from numpy.random import Generator, PCG64

#sys.path.append("/home/zjding/installed_packages/JUST_fiberassign/py/")
sys.path.append("/home/zjding/installed_packages/code_dev/JUST_fiberassign/JUST_fiberassign/") 

In [2]:
from parameters import (
    COLLISION_SEPARATION_ARCSEC,
    COLLISION_SEPARATION_DEG,
    COLLISION_SEPARATION_MM,
    TILE_OUTER_RADIUS_DEG,
    R_PATROL_DEG,
)
from utils import _log, get_fiberpos, write_fba_onetile, find_neighboring_fibers
from fba_single_tile import (
    _degrade_mtl_priorities_on_disk,
    _fits_path_for_tile,
    _healpix_ids_for_disc,
    fba_onetile,
    load_galaxies_from_mtlpix,
    FBASolveFailed,
)

In [3]:
FBA_TIMEOUT_SEC = 300.0  # change this according to the number of iterations


class FBAOnetileTimeout(TimeoutError):
    """Raised when fba_onetile exceeds the per-tile time limit."""


def _run_with_timeout(timeout_sec, func, *args, **kwargs):
    if timeout_sec is None or timeout_sec <= 0:
        return func(*args, **kwargs)

    previous_handler = signal.getsignal(signal.SIGALRM)

    def _handler(signum, frame):
        raise FBAOnetileTimeout(
            f"{func.__name__} exceeded {timeout_sec:.0f}s"
        )

    signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, float(timeout_sec))
    try:
        return func(*args, **kwargs)
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, previous_handler)


def _skipped_tiles_path(out_dir):
    return os.path.join(out_dir, "fba_skipped_tiles.txt")


def _tile_ids_from_file(path):
    if not os.path.isfile(path):
        return set()
    with open(path, encoding="utf-8") as f:
        return {int(line.strip()) for line in f if line.strip()}


def _append_tile_id(tile_id, path):
    """Append one TILEID per line to a persistent tile list."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"{int(tile_id)}\n")
        f.flush()
        os.fsync(f.fileno())


def _fba_onetile_with_timeout(
    tile_ra,
    tile_dec,
    tile_id,
    gal_mtl,
    neighboring_fiber_pairs,
    fiberpos_xy,
    eval_workers,
    skipped_tiles_path,
    timeout_sec=FBA_TIMEOUT_SEC,
    max_iterations=None,
):
    """Run fba_onetile with a per-tile time limit; record skipped tiles on failure."""
    t0 = time.time()
    try:
        result = _run_with_timeout(
            timeout_sec,
            fba_onetile,
            tile_ra,
            tile_dec,
            tile_id,
            gal_mtl,
            neighboring_fiber_pairs,
            fiberpos_xy,
            eval_workers,
            max_iterations,
        )
        if result is None:
            _log(f"  Tile {tile_id}: no assignable targets; skipping")
            return None
        _log(f"  fba_onetile finished in {time.time() - t0:.2f}s")
        return result
    except (FBAOnetileTimeout, FBASolveFailed) as exc:
        _append_tile_id(tile_id, skipped_tiles_path)
        _log(
            f"  Tile {tile_id}: {exc}; "
            f"skipping assignment (recorded in {skipped_tiles_path})"
        )
        return None


def _list_completed_tiles(out_dir):
    completed = set()
    for path in glob.glob(os.path.join(out_dir, "fba_tile_*.fits")):
        base = os.path.basename(path)
        try:
            completed.add(int(base[len("fba_tile_") : -len(".fits")]))
        except ValueError:
            continue
    return completed

In [4]:
class FBAOnetileTimeout(TimeoutError):
    """Raised when fba_onetile_decollided exceeds the per-tile time limit."""

def _run_with_timeout(timeout_sec, func, *args, **kwargs):
    if timeout_sec is None or timeout_sec <= 0:
        return func(*args, **kwargs)

    previous_handler = signal.getsignal(signal.SIGALRM)

    def _handler(signum, frame):
        raise FBAOnetileTimeout(
            f"{func.__name__} exceeded {timeout_sec:.0f}s"
        )

    signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, float(timeout_sec))
    try:
        return func(*args, **kwargs)
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, previous_handler)


def _skipped_tiles_path(out_dir):
    return os.path.join(out_dir, "fba_skipped_tiles.txt")


def _tile_ids_from_file(path):
    if not os.path.isfile(path):
        return set()
    with open(path, encoding="utf-8") as f:
        return {int(line.strip()) for line in f if line.strip()}


def _append_tile_id(tile_id, path):
    """Append one TILEID per line to a persistent tile list."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"{int(tile_id)}\n")
        f.flush()
        os.fsync(f.fileno())


def _record_decollided_fail(tile_id, fail_path):
    """Append a TILEID that timed out in fba_onetile_decollided."""
    _append_tile_id(tile_id, fail_path)



def _list_completed_tiles(out_dir):
    completed = set()
    for path in glob.glob(os.path.join(out_dir, "fba_tile_*.fits")):
        base = os.path.basename(path)
        try:
            completed.add(int(base[len("fba_tile_") : -len(".fits")]))
        except ValueError:
            continue
    return completed

In [5]:
input_mockpath="/home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/galaxy_cluster/input/lightcone_ra_0_90_dec_0_90_rmagcut20.5_cluster_mask.fits"
input_tilepath="/home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/galaxy_cluster/input/tiles_optimized_pass1_2_v0.8.fits"

mock_version="v1"
Npasses = 3
ra0=0
ra1=90
dec0=0
dec1=90
n_workers = 2
eval_workers = 1

##max_iterations = 1
max_iterations = 3

rand_seed = 100
no_resume = True
nside = 32
# set target priority
priority_initial = 100.0
priority_degraded = 2.0

output_fba_path=f"./fba/nwf/max_{max_iterations}iter/"

In [6]:
pool_workers = max(1, n_workers - 2) if n_workers > 2 else n_workers

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

rng = Generator(PCG64(seed=rand_seed))
_log(
    f"rand_seed={rand_seed}, pool_workers={pool_workers}, "
    f"eval_workers={eval_workers}, max_iterations={max_iterations}"
)
_log(f"Patrol radius: {R_PATROL_DEG:.6f} degrees")
_log(
    f"Collision separation: {COLLISION_SEPARATION_ARCSEC} arcsec = "
    f"{COLLISION_SEPARATION_DEG:.6f} deg = {COLLISION_SEPARATION_MM:.3f} mm"
)

fiberpos_xy = get_fiberpos()
N_fibers = fiberpos_xy.shape[0]
_log(f"Number of fibers: {N_fibers}")


input_cat = Table.read(input_mockpath)
mask = (
    (input_cat["ra"] > ra0)
    & (input_cat["ra"] < ra1)
    & (input_cat["dec"] > dec0)
    & (input_cat["dec"] < dec1)
)
input_cat = input_cat[mask]

gal_MTL = Table()

columns = ["TARGETID", "RA", "DEC", "PRIORITY", "SUBPRIORITY"]
for col in columns:
    if col == "TARGETID":
        gal_MTL[col] = input_cat["idx"]
    elif col == "RA":
        gal_MTL[col] = input_cat["ra"]
    elif col == "DEC":
        gal_MTL[col] = input_cat["dec"]
    elif col == "PRIORITY":
        gal_MTL[col] = np.ones(len(input_cat), dtype=float) * priority_initial
        mask = (input_cat["cluster_mask"] == 0)  # distinguish between inside/outside the galaxy cluster center mask
        gal_MTL["PRIORITY"][mask] = priority_degraded
    elif col == "SUBPRIORITY":
        gal_MTL[col] = rng.random(len(input_cat))
    else:
        gal_MTL[col] = input_cat[col]
        
del input_cat
gc.collect()

rand_seed=100, pool_workers=2, eval_workers=1, max_iterations=3
Patrol radius: 0.013021 degrees
Collision separation: 15.625 arcsec = 0.004340 deg = 2.000 mm
Number of fibers: 2184


202

In [7]:
tiles_sub = Table.read(input_tilepath)
##N_tiles = len(tiles_sub)
N_tiles = 11987
# add PASS column to the tiles_sub
tiles_sub["PASS"] = np.array(tiles_sub["TILEID"]//100_000 - 1, dtype=np.int64)

neighboring_fiber_pairs = find_neighboring_fibers(
    fiberpos_xy, patrol_center_separation=12.0
)
_log(f"There are {len(neighboring_fiber_pairs)} paired fibers in one tile")

out_dir = (
    output_fba_path
    + f"/{N_tiles}tiles_{ra0:.1f}ra{ra1:.1f}_{dec0:.1f}dec{dec1:.1f}/seed{rand_seed}/"
)
os.makedirs(out_dir, exist_ok=True)
skipped_tiles_path = _skipped_tiles_path(out_dir)

There are 6312 paired fibers in one tile


In [8]:
if no_resume:
    if os.path.isfile(skipped_tiles_path):
        os.remove(skipped_tiles_path)
_log(f"Skipped tile list: {skipped_tiles_path}")
skipped_tiles = _tile_ids_from_file(skipped_tiles_path)
if skipped_tiles:
    _log(f"Resume: will skip {len(skipped_tiles)} previously failed tile(s)")

nest = False
mtl_dir = out_dir + f"/mtl_nside{nside}/"
mtl_files_exist = bool(glob.glob(os.path.join(mtl_dir, "mtl_healpix_*.fits")))
if no_resume or not mtl_files_exist:
    _log("Split the MTL galaxies into pixels for a fresh fiber-assignment run.")
    npix = hp.nside2npix(nside)
    pix_area_deg2 = hp.nside2pixarea(nside, degrees=True)
    _log(
        f"HEALPix nside={nside}: {npix} pixels, "
        f"~{np.sqrt(pix_area_deg2):.2f} deg/pixel side"
    )

    ra = np.asarray(gal_MTL["RA"], dtype=np.float64)
    dec = np.asarray(gal_MTL["DEC"], dtype=np.float64)
    pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=nest)

    gal_MTL["HEALPIXID"] = pix.astype(np.int64)
    _log(f"Assigned {len(gal_MTL)} galaxies to {len(np.unique(pix))} non-empty pixels")

    sort_idx = np.argsort(pix, kind="stable")
    pix_sorted = pix[sort_idx]
    unique_pix, start_idx, counts = np.unique(
        pix_sorted, return_index=True, return_counts=True
    )

    pixel_cats = {}
    for pix_id, start, count in zip(unique_pix, start_idx, counts):
        rows = sort_idx[start : start + count]
        pixel_cats[int(pix_id)] = gal_MTL[rows]

    _log(f"Built pixel_cats with {len(pixel_cats)} entries")
    _log(
        "Galaxies per pixel: "
        f"min={counts.min()}, median={np.median(counts):.0f}, max={counts.max()}"
    )

    os.makedirs(mtl_dir, exist_ok=True)
    for pix_id, cat_pix in pixel_cats.items():
        ofile = os.path.join(mtl_dir, f"mtl_healpix_{pix_id:05d}.fits")
        cat_pix.write(ofile, overwrite=True)
    _log(f"Wrote {len(pixel_cats)} pixel catalogs to {mtl_dir}")

    del gal_MTL, pixel_cats
    gc.collect()
else:
    _log(f"Resume: reusing existing MTL pixel files in {mtl_dir}")
    del gal_MTL
    gc.collect()

if no_resume:
    completed_tiles = set()
    _log("Resume disabled (--no_resume); starting from scratch")
else:
    completed_tiles = _list_completed_tiles(out_dir)
    if completed_tiles:
        _log(f"Resume: found {len(completed_tiles)} completed tile(s) in {out_dir}")
    else:
        _log(f"Resume: no completed tiles found in {out_dir}")

near_radius_deg = 1.2 * TILE_OUTER_RADIUS_DEG
_log(
    f"Load the sample from healpixID files, where some galaxies are within "
    f"{near_radius_deg:.6f} deg of the tile center."
)

Skipped tile list: ./fba/nwf/max_3iter//11987tiles_0.0ra90.0_0.0dec90.0/seed100/fba_skipped_tiles.txt
Split the MTL galaxies into pixels for a fresh fiber-assignment run.
HEALPix nside=32: 12288 pixels, ~1.83 deg/pixel side
Assigned 13032492 galaxies to 1568 non-empty pixels
Built pixel_cats with 1568 entries
Galaxies per pixel: min=3418, median=8422, max=10830
Wrote 1568 pixel catalogs to ./fba/nwf/max_3iter//11987tiles_0.0ra90.0_0.0dec90.0/seed100//mtl_nside32/
Resume disabled (--no_resume); starting from scratch
Load the sample from healpixID files, where some galaxies are within 0.716160 deg of the tile center.


In [10]:
#for passid in range(Npasses):
passid = 1

Ntiles = 20  # number of tiles to run

t0 = time.time()
_log(f"passid: {passid}")
mask = (tiles_sub["PASS"] == passid)
tiles_ra_pass = np.asarray(tiles_sub["RA_NEW"][mask], dtype=np.float64)
tiles_dec_pass = np.asarray(tiles_sub["DEC_NEW"][mask], dtype=np.float64)
tiles_id_pass = np.asarray(tiles_sub["TILEID"][mask], dtype=np.int64)
_log(f"Number of tiles in pass {passid}: {len(tiles_ra_pass)}")

for tile_ra, tile_dec, tile_id in zip(
    tiles_ra_pass[0:Ntiles], tiles_dec_pass[0:Ntiles], tiles_id_pass[0:Ntiles]
):
    out_fits = _fits_path_for_tile(out_dir, tile_id)
    if not no_resume and tile_id in completed_tiles:
        _log(f"Tile {tile_id}: skip (already exists) {out_fits}")
        continue
    if not no_resume and tile_id in skipped_tiles:
        _log(
            f"Tile {tile_id}: skip (assignment previously failed; "
            f"see {skipped_tiles_path})"
        )
        continue

    pix_ids = _healpix_ids_for_disc(
        tile_ra, tile_dec, near_radius_deg, nside, nest=nest
    )
    gal_mtl = load_galaxies_from_mtlpix(
        tile_ra,
        tile_dec,
        mtl_dir,
        near_radius_deg,
        nside=nside,
        nest=nest,
    )
    _log(
        f"Tile {tile_id} ({tile_ra:.4f}, {tile_dec:.4f}): "
        f"loaded {len(gal_mtl)} galaxies from {len(pix_ids)} HEALPix file(s) "
        f"pix={pix_ids.tolist()}"
    )

    t_fba_start = time.time()
    fba_out = _fba_onetile_with_timeout(
        tile_ra,
        tile_dec,
        tile_id,
        gal_mtl,
        neighboring_fiber_pairs,
        fiberpos_xy,
        eval_workers,
        skipped_tiles_path,
        timeout_sec=FBA_TIMEOUT_SEC,
        max_iterations=max_iterations
    )
    t_fba_end = time.time()
    _log(f"  fba total time: {t_fba_end - t_fba_start:.2f}s")
    if fba_out is None:
        skipped_tiles.add(int(tile_id))
        _log(f"Tile {tile_id}: assignment skipped\n")
        _log("=====================")
        continue

    fba_result, targets_id_list_alltiles = fba_out

    per_tile_assigned = defaultdict(lambda: {"target_ids": [], "fiber_ids": []})
    for _tid, _node in fba_result["target_to_fiber"].items():
        _node = str(_node)
        if "_fiber_" not in _node:
            continue
        _tile, _fid = _node.rsplit("_fiber_", 1)
        try:
            _fid = int(_fid)
        except Exception:
            continue
        per_tile_assigned[_tile]["target_ids"].append(int(_tid))
        per_tile_assigned[_tile]["fiber_ids"].append(_fid)

    assigned_tarids_all = []
    for _tile, _vals in per_tile_assigned.items():
        assigned_targets_id = _vals["target_ids"]
        assigned_tarids_all.extend(assigned_targets_id)
        write_fba_onetile(
            tile_id=_tile,
            assigned_targets_id=assigned_targets_id,
            assigned_fiber_id=_vals["fiber_ids"],
            targets_id_list_alltiles=targets_id_list_alltiles,
            out_fits_path=out_fits,
            overwrite=True,
        )
    t_outfits_end = time.time()
    _log(
        f"  Wrote {out_fits} ({len(assigned_targets_id)} assignments); "
        f"takes {t_outfits_end - t_fba_end:.2f}s"
    )

    if assigned_tarids_all:
        affected_pix = np.unique(
            gal_mtl["HEALPIXID"][
                np.isin(gal_mtl["TARGETID"], assigned_tarids_all)
            ]
        )
        n_disk = _degrade_mtl_priorities_on_disk(
            mtl_dir, assigned_tarids_all, priority_degraded, affected_pix
        )
        _log(
            f"  Updated PRIORITY on disk for {n_disk} row(s) "
            f"in {len(affected_pix)} pixel file(s): {affected_pix.tolist()}"
        )
    completed_tiles.add(int(tile_id))
    _log(f"  Update the MTL PRIORITY files in {time.time() - t_outfits_end:.2f}s\n")
    _log("=====================")

_log(f"pass {passid} running time (s): {time.time() - t0:.1f}")

if os.path.isfile(skipped_tiles_path):
    n_skipped = len(_tile_ids_from_file(skipped_tiles_path))
    _log(f"Total tiles skipped after fallback failure: {n_skipped} ({skipped_tiles_path})")

passid: 1
Number of tiles in pass 1: 3996
Tile 208445 (65.4608, 86.6870): loaded 25300 galaxies from 3 HEALPix file(s) pix=[5, 13, 14]
  Tile 208445: 120 fiber-pair constraints, 203 target pairs
  Iteration 1/3: violations=26, unique_opts=52, eval_workers=1
  Iteration 2/3: violations=8, unique_opts=16, eval_workers=1
  solve_tile_group finished: cost=12356395948.00, assigned=1598, used_fibers=1598/2184

=== Single Tile/Group Complete ===
All constraints satisfied.
  fba_onetile finished in 27.76s
  fba total time: 27.76s
  Wrote ./fba/nwf/max_3iter//11987tiles_0.0ra90.0_0.0dec90.0/seed100/fba_tile_208445.fits (1598 assignments); takes 0.01s
  Updated PRIORITY on disk for 1598 row(s) in 2 pixel file(s): [5, 14]
  Update the MTL PRIORITY files in 0.03s

Tile 208748 (86.5695, 87.5306): loaded 25930 galaxies from 6 HEALPix file(s) pix=[0, 1, 5, 6, 14, 15]
  Tile 208748: 58 fiber-pair constraints, 90 target pairs
  Iteration 1/3: violations=14, unique_opts=28, eval_workers=1
  Iteration 2/